# 07 - Ambiguous question: the Coordinator asks before it searches

**Scenario.** A customer writes *"My phone isn't charging properly, what should I do?"* - four phone families, many
models, several charging behaviours. Instead of guessing, the planner returns a **clarifying question** and the graph
**interrupts**. The answer resumes the same thread from its checkpoint.

**Prerequisites:** `.env` with `OPENAI_API_KEY`, `JUDGE_OPENAI_API_KEY` and `SERPER_API_KEY`; the vector stores (`make ingest`); the CRM & Support MCP server (`make mcp-server-up`).

In [ ]:
# Setup: make src/ importable and load the settings from .env
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*langchain.mcp.*")

from agentic_rag.config import AgentSettings

settings = AgentSettings.load(ROOT)
print(f"Coordinator: {settings.model} ({settings.coordinator_effort}) | specialists: {settings.specialist_model} "
      f"({settings.specialist_effort}) | judge: {settings.judge_model} (pass >= {settings.judge_pass_score})")

In [ ]:
# Start the multi-agent service: SQLite memory + CRM MCP tools + 5 specialists + Coordinator graph
from agentic_rag.graph import CoordinatorService

service = await CoordinatorService.create(settings)
print(service.crm.describe())
print(service.roster())

In [ ]:
result = await service.ask("My phone isn't charging properly, what should I do?", user_id="cust-web-chat",
                           on_event=print)
result

## Answer the clarification(s)
The planner may ask up to two questions; prepared answers are used in order.

In [ ]:
ANSWERS = ["It's a Zentrix Z9 Plus. It charges very slowly with the adapter that came in the box.",
           "No error message is shown."]
for answer in ANSWERS:
    if result.status != "needs_clarification":
        break
    print("Assistant asked:", result.interrupt["question"])
    print("User answers   :", answer)
    result = await service.resume(result.thread_id, answer, on_event=print)
result

In [ ]:
# The clarification is part of the thread's conversation (short-term memory)
for role, text in await service.history(result.thread_id):
    print(f"{role:>5}: {text[:160]}")

In [ ]:
# Close the memory databases
await service.aclose()

## What to observe
- No specialist ran before the clarification - nothing is wasted on guesses.
- After the answer, the plan pins the exact model (`Z9 Plus`).
- The interrupt survives restarts: the thread is stored in `data/agent_state/checkpoints.sqlite`.